# DADA2: 04 — Evaluation

Three TODOs.

You will conclude the assignment by running a RAG eval.

Three questions to answer with numbers:
- Is the decision model as good a router as the chat model? 
- Does the diagnosis agent beat the ladder, and at what cost?
- which parts of what you built were worth building?

In [1]:
from dotenv import load_dotenv
load_dotenv()

import time
from rag import config, decide, evaluate, retrieval, agent, ladder

## Read this first

This notebook is where the Gemini quota goes. The ladder agent spends about
five chat calls per question and yours about three, and the judge spends one
more on each answer. Fourteen questions through both agents is roughly 150
calls against a limit of 500 a day.

So, you will want to run it once, when notebooks 02 and 03 already pass their final cells. The comparison cells sleep between questions on purpose. Start them and leave them alone.

In [2]:
specific = evaluate.load_evalset("specific")
comparative = evaluate.load_evalset("comparative")
items = specific + comparative

print(f"{len(specific)} specific + {len(comparative)} comparative = {len(items)} labeled questions")
print("reranker:       cohere")
print("chat model:    ", config.CHAT_MODEL)
print("decision model:", config.DECISION_MODEL)

14 specific + 10 comparative = 24 labeled questions
reranker:       cohere
chat model:     gemini-3.5-flash-lite
decision model: clef-flash


## 1. Two routers

### TODO 4.1 — the decision model against the chat model (1 pt)

Every labeled question has a kind: the 14 in `evalset.json` are specific,
the 10 in `evalset_hard.json` are comparative. Ask both models which kind each
one is, and keep the time.

Write the chat-model version yourself. It is a `with_structured_output` call
with a `Literal` field, the same thing you did for `Filter` in notebook 02.
Use `decide.SCOPES` for the descriptions so both models are asked the same
question.

**Steps:**

1. Define a scoper that uses Scope to get structured output from chat model
3. In the loop below, the decision-model line is written. Add the same two lines for the chat model, keyed `"chat"`: time the call, add the seconds, add one to `right` when the choice matches `kind`.


In [3]:
from typing import Literal
from pydantic import BaseModel, Field
from rag.util import retry

# TODO 4.1 — a scoper that uses Scope to get structured output from chat model
class Scope(BaseModel):
    """Which kind of question this is."""
    scope: Literal["specific", "comparative", "figure"] = Field(
        description="; ".join(f"{k}: {v}" for k, v in decide.SCOPES.items()))

def chat_scope(question):
    t = time.time()
    out = retry(scoper.invoke, "Question about SEC 10-K filings: " + question
                + "\nWhat does it ask for?")
    return out.scope, time.time() - t

#TODO 4.1
# your code here
scoper = config.chat().with_structured_output(Scope)

In [4]:
labeled = [(i["question"], "specific") for i in specific] + \
           [(i["question"], "comparative") for i in comparative]

right = {"decision": 0, "chat": 0}
secs = {"decision": 0.0, "chat": 0.0}
for q, kind in labeled:
    t = time.time(); choice, _ = decide.scope(q); secs["decision"] += time.time() - t
    right["decision"] += (choice == kind)
    t = time.time(); choice, _ = chat_scope(q); secs["chat"] += time.time() - t
    right["chat"] += (choice == kind)
    time.sleep(3)

n = len(labeled)
for name in ("decision", "chat"):
    print(f"{name:9} {right[name]}/{n} right, {secs[name]/n:.2f}s per call")

decision  20/24 right, 0.85s per call
chat      19/24 right, 0.72s per call


NOTE: Accuracy is close, and the chat model is probably not
slower. If you expected the decision model to win on both, write down what it does give you instead: a probability per option, and a call that does not count against the daily limit. Whether those are worth having is the question
the next section answers.

## 2. Two agents

### TODO 4.2 — the ladder against the diagnosis (1 pt)

`evaluate.compare_agents` runs each agent over the questions, has the chat
model judge every answer, and keeps the bill: chat calls, decision calls, and
seconds per question. Both agents expose `run(question)` returning the final
state, which is all it needs.

Fourteen questions is enough. Use eight specific and six comparative so both
kinds are in the mix.

**Steps:** nothing to write. Run the cell once, when both agents pass their final cells. If a 429 cuts it off, score the two agents in separate cells rather than restarting the pair.

This will take a while. Plan accordingly. 

Make sure your rate limit is reset or use a different provider (e.g., Groq, OpenRouter) instead of the Gemini API. 

Let it run and come back to see your results.


In [5]:
# TODO 4.2
subset = specific[:8] + comparative[:6]

rows = evaluate.compare_agents({"ladder": ladder.run, "diagnosis": agent.run}, subset)

  agent         accuracy  chat/q  decision/q    s/q
  --------------------------------------------------


c:\Users\harsh\AgenticAI_Fall26\DADA2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ladder            0.93     4.1         0.0   13.0
  diagnosis         0.79     2.6         2.3    9.4


In [6]:
# the judge's reasons, on anything either agent got wrong
_, notes = evaluate.score_agent(agent.run, subset[:4])
for note in notes:
    mark = "ok " if note["correct"] else "WRONG"
    print(f"[{mark}] {note['question'][:66]}  ({note['chat_calls']} chat, {note['decision_calls']} decision)")
    print(f"        {note['reason'][:150]}\n")

[ok ] What does MU state that the preparation of financial statements an  (2 chat, 2 decision)
        CORRECT
The answer accurately reflects the standard U.S. GAAP requirement for management regarding estimates and judgments as typically stated in SEC 

[ok ] What termination fee is AMD required to pay ZT Systems under certa  (2 chat, 2 decision)
        CORRECT
The provided answer accurately states the termination fee amount and circumstances mentioned in the SEC filings regarding the AMD and ZT Syste

[ok ] What factors does NVIDIA believe are the principal competitive fac  (2 chat, 2 decision)
        CORRECT
The answer accurately and comprehensively lists the principal competitive factors for NVIDIA as stated in the specified fiscal year filing.

[ok ] What competitive pressures from Intel and Nvidia did AMD face in f  (2 chat, 2 decision)
        CORRECT

The answer accurately outlines the competitive pressures AMD faced from Intel and Nvidia in fiscal year 2025 as described in t

In [7]:

   _, notes = evaluate.score_agent(agent.run, subset[4:])
   for note in notes:
       if not note["correct"]:
           print(note["question"])
           print("   ", note["reason"][:200], "\n")
   print("done")

What financial charge did NVIDIA incur in fiscal year 2026 due to export requirements for its H20 product, and what was the cause?
    INCORRECT
Reason: The provided answer hallucinates financial figures, events, and future fiscal year details that do not exist in standard SEC filings (as fiscal year 2026 for NVIDIA has not occurred  

What target markets does NVIDIA include in its business for fiscal year 2024?
    INCORRECT. The provided passages (in standard NVIDIA 10-K filings for fiscal year 2024) clearly detail NVIDIA's target markets and reportable segments such as Accelerated Computing and Graphics, inclu 

done


If the diagnosis agent got something wrong that the ladder got right, find
out which half failed. Run `decide.scope` and `decide.diagnose` on that
question by hand and look at the probabilities. A wrong route and a wrong
verdict are different bugs, and the fix for one is a threshold while the fix
for the other is a search strategy.

### TODO 4.3 — write it up (2 pts)

In the cell below, in your own words. Four or five short paragraphs.

**Conditions.** Reranker, chat model, decision model, k, chunk size. Enough
for someone to re-run you.

**The routers.** Which was more accurate, which was faster, and by how much.
Then say what the decision model is for, given those numbers.

**The agents.** Accuracy, chat calls per question, seconds per question, for
both. Name the question where they differed most and say why.

**One thing that did not help.** Something you built that the numbers say you
could remove. Say what the mechanism is, not that the model is better at it.

**What you would ship**, and what you would want to know about your users'
questions first.

Two things that cost marks: an improvement the table does not show, and a
difference explained by "the model is better" when the reason is in your
code.

**Steps:** write it as prose, four or five short paragraphs. The bold headings in the cell below are the order; replace the cell's text under each. Every number you quote should be in a table above.


---

### Your write-up

**Conditions.** Chunks of 1,200 characters with 150 overlap (2,124 chunks from 27 section files), embedded with qwen3-embedding-0.6b on Cloudflare Workers AI (1,024 dimensions) into the Pinecone index `dada2-1024`. Retrieval returns k=5 passages; the reranker is Cohere `rerank-v3.5` over a pool of 10; RRF uses k=60. The chat model is `gemini-3.5-flash-lite`, used for filter extraction, query rewrites, planning, answers and judging. The decision model is Cloudflare's `clef-flash`. The router comparison used all 24 labeled questions; the agent comparison used 14 (8 specific, 6 comparative).

**The routers.** The decision model got 20/24 right at 0.85 s per call; the chat model got 19/24 at 0.72 s. That is one question of accuracy in the decision model's favour and 0.13 s of speed in the chat model's, so neither result is a meaningful win. What the decision model is for is not accuracy or speed: it returns a probability for every option, which is what lets `route` fall back to `filter+rrf` below 0.6 and `diagnose` refuse an `absent` below 0.8. It also costs nothing against the 500-a-day Gemini limit, which mattered in practice in this assignment.

**The agents.** The ladder scored 0.93 (13/14) at 4.1 chat calls and 13.0 s per question. The diagnosis agent scored 0.79 (11/14) at 2.6 chat calls, 2.3 decision calls and 9.4 s per question: about 37% fewer chat calls and 28% less time. Re-judging the diagnosis agent on the last ten questions found two misses, not three, so on 14 questions the judge itself moves the score by about one question. The two misses are different in kind. On NVIDIA's fiscal 2026 H20 charge, the judge's reason was that fiscal 2026 "has not occurred": it graded against its own training knowledge instead of the filing, which is a flaw in the evaluation rather than the agent. On NVIDIA's fiscal 2024 target markets, the answer said the passages did not cover the topic, though the business section does. The mechanism is in my code: the diagnosis agent takes the first stopping verdict after one filtered search and only searches again if the verdict says to, while the ladder climbs to a stronger search on every failure. Spending fewer searches is the saving, and a stop after one weak search is its price.

**What did not help.** Multi-query fusion (`filter+rrf`). In notebook 02's sample, the filter alone scored recall [R1] and MRR [M1], and fusion scored [R2] and [M2], for an extra chat call per question. The reason is the filter: once it restricts the search to one filing, every rewrite searches the same few dozen chunks, so the lists mostly agree, and RRF can only reshuffle passages that were already found, which can move the one right passage down a place.

**What I would ship.** The diagnosis agent, with two changes. First, accept `sufficient` only above a probability threshold and send anything below it to `stronger`, the way `absent` is already second-guessed, so a single weak search no longer ends the question. Second, give the judge the gold file instead of letting it grade from memory, since a judge that does not believe fiscal 2026 exists cannot score questions about it. Before setting the threshold I would want to know the mix of my users' questions: the filter wins on questions naming one filing and loses on comparative ones, so the share of comparative questions decides whether early stopping is a cheap shortcut or the main source of wrong answers.

**Copy your write-up to README.md.**